# Train CUT on horse2zebra (Google Colab)

Trains [Contrastive Unpaired Translation](https://github.com/taesungp/contrastive-unpaired-translation) on the `horse2zebra` dataset.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine; A100/L4 faster).

The full paper config is 400 epochs (`n_epochs=200`, `n_epochs_decay=200`) which takes many hours on a T4. The cells below default to a much shorter demo run; change `N_EPOCHS` / `N_EPOCHS_DECAY` for a real training run.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. (Optional) Mount Google Drive for persistent checkpoints

Colab VMs are ephemeral. Mounting Drive lets you resume training and keep results across sessions. Skip this cell if you don't care about persistence.

In [ ]:
USE_DRIVE = False  # set True to persist checkpoints to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINTS_DIR = '/content/drive/MyDrive/phd/models/cut_h2z_checkpoints'
    RESULTS_DIR = '/content/drive/MyDrive/phd/models/cut_h2z_results'
else:
    CHECKPOINTS_DIR = '/content/checkpoints'
    RESULTS_DIR = '/content/results'

import os
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('checkpoints ->', CHECKPOINTS_DIR)
print('results     ->', RESULTS_DIR)

## 3. Clone the repo

In [ ]:
%cd /content
![ -d contrastive-unpaired-translation ] || git clone https://github.com/valerybr/contrastive-unpaired-translation.git
%cd /content/contrastive-unpaired-translation

## 4. Install dependencies

Colab already ships PyTorch + torchvision; we only need the repo's lightweight extras.

In [ ]:
!pip install -q dominate visdom GPUtil packaging

## 5. Patch a pre-existing argparse bug

Upstream `models/cut_model.py` declares `choices='(CUT, cut, FastCUT, fastcut)'` — a *string*, which argparse treats as a set of characters. Passing `--CUT_mode CUT` therefore fails as "invalid choice". The default works, but the launcher commands below pass the flag explicitly (and we may want FastCUT), so fix it.

In [ ]:
import re, pathlib
p = pathlib.Path('models/cut_model.py')
src = p.read_text()
fixed = src.replace(
    "choices='(CUT, cut, FastCUT, fastcut)'",
    "choices=('CUT', 'cut', 'FastCUT', 'fastcut')",
)
if fixed != src:
    p.write_text(fixed)
    print('patched')
else:
    print('already patched or upstream changed')

## 6. Download horse2zebra

~117 MB. Contains `trainA/` (horses), `trainB/` (zebras), `testA/`, `testB/`.

In [ ]:
!bash ./datasets/download_cut_dataset.sh horse2zebra
!ls datasets/horse2zebra && echo '---' && ls datasets/horse2zebra/trainA | head -3 && ls datasets/horse2zebra/trainA | wc -l

## 7. Train

Key flags:
- `--CUT_mode CUT` — full CUT (identity NCE, `lambda_NCE=1.0`). Use `FastCUT` for the faster variant.
- `--display_id -1` — disable visdom (Colab can't easily host the visdom server).
- `--num_threads 2` — Colab free tiers have 2 vCPUs.
- `--save_epoch_freq` — how often checkpoints are written (every N epochs).
- `--n_epochs` / `--n_epochs_decay` — constant-LR epochs, then linearly-decayed epochs. Paper: 200 + 200.

**Resume** a previous run by adding `--continue_train --epoch_count <next_epoch>`.

In [ ]:
EXP_NAME = 'horse2zebra_CUT'
N_EPOCHS = 5          # constant-LR epochs (paper: 200)
N_EPOCHS_DECAY = 5    # decay epochs         (paper: 200)
SAVE_EPOCH_FREQ = 2

cmd = f"""python train.py \
  --dataroot ./datasets/horse2zebra \
  --name {EXP_NAME} \
  --CUT_mode CUT \
  --checkpoints_dir {CHECKPOINTS_DIR} \
  --n_epochs {N_EPOCHS} \
  --n_epochs_decay {N_EPOCHS_DECAY} \
  --save_epoch_freq {SAVE_EPOCH_FREQ} \
  --display_id -1 \
  --num_threads 2"""
print(cmd)
!{cmd}

## 8. Inference on the test set

`test.py` loads the generator from `<checkpoints_dir>/<name>/latest_net_G.pth` and writes an HTML gallery to `<results_dir>/<name>/test_latest/index.html`.

In [ ]:
cmd = f"""python test.py \
  --dataroot ./datasets/horse2zebra \
  --name {EXP_NAME} \
  --CUT_mode CUT \
  --checkpoints_dir {CHECKPOINTS_DIR} \
  --results_dir {RESULTS_DIR} \
  --phase test \
  --num_test 20"""
print(cmd)
!{cmd}

## 9. Preview generated images

In [ ]:
import glob, os
from IPython.display import Image, display, HTML

img_dir = os.path.join(RESULTS_DIR, EXP_NAME, 'test_latest', 'images')
reals = sorted(glob.glob(os.path.join(img_dir, '*_real_A.png')))[:6]
for real in reals:
    fake = real.replace('_real_A.png', '_fake_B.png')
    display(HTML(f"<h4>{os.path.basename(real).replace('_real_A.png','')}</h4>"))
    display(Image(real, width=256), Image(fake, width=256))

## 10. (Optional) Train FastCUT instead

FastCUT skips the identity NCE, uses `lambda_NCE=10.0` and flip-equivariance. Lighter and ~2× faster than CUT.

Its `modify_commandline_options` overrides `n_epochs=150, n_epochs_decay=50` by default — override on the CLI to shorten for a demo.

In [ ]:
# Uncomment to run
# !python train.py \
#   --dataroot ./datasets/horse2zebra \
#   --name horse2zebra_FastCUT \
#   --CUT_mode FastCUT \
#   --checkpoints_dir {CHECKPOINTS_DIR} \
#   --n_epochs 5 --n_epochs_decay 5 \
#   --display_id -1 --num_threads 2